# Schema — add `contributors` to the task layerAdds a free-text field to the task layer in `datateam_portfolio_v2` sotasks can carry a comma-separated list of secondary contributorsalongside the existing `assignee` (the primary owner).**Run this once, before deploying the contributors release.** Idempotent —re-running is a no-op once the field exists.You must be **owner or admin** of `datateam_portfolio_v2`.**Field shape**- Name: `contributors`- Type: `esriFieldTypeString`, length 1000 (room for ~12-15 names with commas)- Nullable: yes (existing records will be NULL until populated via the  in-app form or the console-command helper)This mirrors the way projects already carry secondary members — projectshave `contact` (the Lead) + `other_members` (comma-separated). Tasks nowget the same pattern: `assignee` (primary owner) + `contributors`(secondary helpers).

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

gis = GIS("home")
print(f"Signed in as {gis.users.me.username} @ {gis.url}")

TASK_LAYER_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/1"
tasks = FeatureLayer(TASK_LAYER_URL, gis)
print("Task layer:", tasks.properties.get('name', '(unknown)'))

## Step 1 — Check whether the field already exists

In [ ]:
existing_fields = [f['name'] for f in tasks.properties.fields]
print(f"Total fields: {len(existing_fields)}")
already_exists = 'contributors' in existing_fields
if already_exists:
    print("✓ contributors is already in the schema — nothing to add.")
else:
    print("ℹ contributors NOT in the schema — Step 2 will add it.")

## Step 2 — Add the fieldOnly runs if the field doesn't exist.

In [ ]:
if already_exists:
    print("Skipped — already present.")
else:
    field_def = {
        "name": "contributors",
        "type": "esriFieldTypeString",
        "alias": "Contributors",
        "length": 1000,
        "nullable": True,
        "editable": True,
        "defaultValue": None
    }
    result = tasks.manager.add_to_definition({"fields": [field_def]})
    print("add_to_definition result:", result)

## Step 3 — Verify

In [ ]:
tasks_fresh = FeatureLayer(TASK_LAYER_URL, gis)
fields_now = [f['name'] for f in tasks_fresh.properties.fields]
if 'contributors' in fields_now:
    print("✓ contributors is in the schema.")
    with_any = tasks_fresh.query(where="contributors IS NOT NULL AND contributors <> ''", out_fields="OBJECTID", return_count_only=True)
    print(f"  Tasks currently carrying contributors: {with_any}")
    print()
    print("Next: deploy the app code (v1.76.0.0) and use the in-form picker")
    print("to add contributors. The console helper findContributorCandidates()")
    print("scans time_entries for plausible suggestions.")
else:
    print("✗ contributors NOT in the schema — check the Step 2 result for errors.")

## Rollback (if needed)To remove the field again:    tasks.manager.delete_from_definition({"fields": [{"name": "contributors"}]})Deletes the column and any data in it.